# ДЗ 2. Дообучение seq2seq-модели для суммаризации новостей

---

### Зачем это нужно

Суммаризация — это одна из задач преобразования текста, с которой часто сталкиваются NLP-инженеры. На вход модели поступает длинный текст, а на выходе требуется получить его краткое и информативное изложение. Для решения этой задачи используется архитектура encoder-decoder с механизмом cross-attention. Та же архитектура лежит в основе машинного перевода, перефразирования, исправления текста и многих других задач преобразования текста. Разобравшись с ее работой на примере суммаризации, вы поймете общий принцип, который затем переносится на широкий класс генеративных задач.

Не менее важный навык — корректно оценивать качество генерации. В задачах классификации существует единственный правильный ответ, поэтому качество удобно измерять такими метриками, как Accuracy или F1. В задачах генерации ситуация иная: одну и ту же мысль можно выразить множеством разных способов. Поэтому здесь используются специальные метрики, например ROUGE и BLEU, однако их значения не всегда совпадают с человеческой оценкой качества. В этом задании вы не только научитесь вычислять эти метрики, но и поймете их ограничения, чтобы уметь правильно интерпретировать результаты моделей.

### Что вы сделаете

1. Загрузите датасет и подготовите его для обучения seq2seq-модели (блок 1).
2. Получите бейзлайновые результаты: ROUGE/BLEU для дообученной референсной модели и для базовой модели без дообучения (блок 2).
3. Дообучите модель `ruT5-base` на подвыборке и проанализируете, как изменится качество генерации (блок 3).
4. Сравните результаты всех моделей в единой таблице и на конкретных примерах разберете ситуации, в которых автоматические метрики расходятся с человеческой оценкой (блок 4).
5. Сформулируете выводы по результатам эксперимента и объясните, какую роль играет механизм cross-attention в работе модели (блок 5).

> **Сколько займет:** ориентировочно 4—6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU. На CPU обучение модели будет занимать слишком много времени. Если GPU недоступен, вы все равно сможете выполнить блоки 1—2 и 4, однако блок 3 (дообучение модели) рассчитан на использование GPU.

### Оценивание

Максимальный балл за работу — 10. Баллы за каждый блок указаны в его заголовке.

## Настройка окружения

In [1]:
!pip install transformers datasets sentencepiece rouge_score sacrebleu accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=d1f07bd6fd4f5b2214bdc8ca45ed2661a895b387c6c54dafb62d22360e866fec
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [2]:
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from IPython.display import display

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from datasets import Dataset, DatasetDict, load_dataset
from rouge_score import rouge_scorer
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
set_seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", None)

print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")


device: cuda


## 1. Данные и токенизация для seq2seq (1.5 балла)

**Датасет:** [`IlyaGusev/gazeta`](https://huggingface.co/datasets/IlyaGusev/gazeta) содержит новости на русском языке: статья `text` и ее краткое саммари `summary`. Ячейка ниже грузит датасет.

**Пояснение.** В моделях seq2seq энкодер сначала преобразует входную статью в последовательность контекстных представлений. Затем декодер генерирует саммари токен за токеном, а механизм cross-attention на каждом шаге позволяет ему обращаться к представлениям, построенным энкодером, и выбирать, какая информация из исходного текста наиболее важна для генерации следующего токена.

Поэтому при подготовке данных необходимо сформировать две последовательности: токенизированный вход (статью) и токенизированную целевую последовательность (саммари). Целевая последовательность записывается в специальное поле `labels` — именно по ней во время обучения вычисляется loss.

Важно учитывать, что для входной и целевой последовательностей обычно задаются разные максимальные длины (`max_length`): статьи, как правило, значительно длиннее своих кратких саммари, поэтому ограничения на число токенов для них различаются.

In [3]:
BASE_SEQ2SEQ  = "ai-forever/ruT5-base" # базовая модель для дообучения
REF_SUMMARIZER = "IlyaGusev/rut5_base_sum_gazeta" # уже дообученная (в качестве сильного ориентира)

N_TRAIN, N_VAL, N_TEST = 3000, 300, 300 # подвыборка (уменьшите, если не хватает памяти/времени)

def _load_gazeta():
    """Грузит gazeta. В свежих версиях datasets (>=3.0) загрузочные скрипты
    больше не поддерживаются, поэтому пробуем несколько вариантов."""
    for kwargs in ({}, {"revision": "v2.0"}, {"revision": "refs/convert/parquet"}):
        try:
            return load_dataset("IlyaGusev/gazeta", **kwargs)
        except Exception as e:
            print(f"load_dataset(**{kwargs}) не сработал: {type(e).__name__}: {e}")
    raise RuntimeError("Не удалось загрузить IlyaGusev/gazeta")

def load_summarization_data():
    """Возвращает DatasetDict со сплитами train/val/test"""
    ds = _load_gazeta()

    def take(split, n):
        # фиксированный seed -> один и тот же срез при каждом запуске
        d = ds[split].shuffle(seed=RANDOM_STATE).select(range(min(n, len(ds[split]))))
        return d.remove_columns([c for c in d.column_names if c not in ("text", "summary")])

    out = DatasetDict(train=take("train", N_TRAIN),
                      validation=take("validation", N_VAL),
                      test=take("test", N_TEST))
    print(f'train={len(out["train"])}, val={len(out["validation"])}, test={len(out["test"])}')
    return out

raw = load_summarization_data()

test_texts = list(raw["test"]["text"])
test_refs = list(raw["test"]["summary"])
print("\nПример пары:")
print("СТАТЬЯ :", test_texts[0][:200])
print("САММАРИ :", test_refs[0][:200])


README.md:   0%|          | 0.00/13.4k [00:00<?, ?B/s]

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  252MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/train/0001.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

default/train/0001.parquet: downloading bytes:           |  0.00B            

...

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

train=3000, val=300, test=300

Пример пары:
СТАТЬЯ : Американские вооруженные силы должны быть готовы вернуться в Афганистан при необходимости. Об этом заявил экс-президент США Дональд Трамп , выступая перед своими сторонниками на митинге в штате Алабам
САММАРИ : Экс-президент США продолжает критиковать администрацию Джо Байдена за действия в Афганистане. На митинге в Алабаме Трамп заявил, что американские войска должны быть готовы вернуться в страну — по слов


In [4]:
# Токенизатор базовой модели и лимиты длины
s2s_tok = AutoTokenizer.from_pretrained(BASE_SEQ2SEQ)
MAX_SRC, MAX_TGT = 512, 160

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.00MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

### ✍️ Задание 1.1. Препроцессинг для seq2seq

Реализуйте функцию `preprocess(batch)`, которая токенизирует батч примеров:

- вход — `batch["text"]` с `max_length=MAX_SRC` и `truncation=True`;
- таргет — `batch["summary"]` с `max_length=MAX_TGT` и `truncation=True`, ее id нужно положить в ключ `"labels"` результата.

Затем примените ее ко всем сплитам через `.map(preprocess, batched=True, remove_columns=raw["train"].column_names)` и сохраните в переменную `tokenized`.

In [5]:
# ✍️ ВАШ КОД (задание 1.1)
def preprocess(batch):
    """Токенизирует статью (вход энкодера) и саммари (labels для декодера)."""
    model_inputs = s2s_tok(
        batch["text"],
        max_length=MAX_SRC,      # статья длинная -> лимит больше
        truncation=True,         # обрезаем всё, что длиннее лимита
    )
    labels = s2s_tok(
        text_target=batch["summary"],  # целевая последовательность
        max_length=MAX_TGT,            # саммари короткое -> лимит меньше
        truncation=True,
    )
    # именно по полю "labels" Trainer считает cross-entropy loss
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = raw.map(preprocess, batched=True,
                    remove_columns=raw["train"].column_names)

print(tokenized)
# sanity check: длины не превышают лимиты, labels на месте
ex = tokenized["train"][0]
print("input_ids:", len(ex["input_ids"]), "| labels:", len(ex["labels"]))
print("decoded labels:", s2s_tok.decode(ex["labels"], skip_special_tokens=True)[:200])

# распределение длин в токенах: показывает, какая доля примеров реально обрезается
src_len = [len(x) for x in tokenized["train"]["input_ids"]]
tgt_len = [len(x) for x in tokenized["train"]["labels"]]
print(f"src: median={np.median(src_len):.0f}, доля обрезанных={np.mean(np.array(src_len) >= MAX_SRC):.1%}")
print(f"tgt: median={np.median(tgt_len):.0f}, доля обрезанных={np.mean(np.array(tgt_len) >= MAX_TGT):.1%}")


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300
    })
})
input_ids: 512 | labels: 81
decoded labels: Департамент транспорта Москвы проведет внеплановые проверки автопарков частных перевозчиков, проведет техосмотр автобусов и медосмотр водителей после ДТП с автобусом в подземном переходе около станции
src: median=512, доля обрезанных=98.9%
tgt: median=69, доля обрезанных=0.0%


### 💬 Задание 1.2. Вопрос на понимание (впишите ответ ниже)

Ответьте в 2—4 предложениях: почему у source (`MAX_SRC=512`) и у target (`MAX_TGT=160`) разные максимальные длины? Что произойдет, если поставить `MAX_SRC` слишком маленьким - например, 64?

> *Ваш ответ:* Вход и выход у суммаризации сильно асимметричны, и это видно по замерам на моей подвыборке: медианная длина саммари — 69 токенов, при `MAX_TGT=160` не обрезается ни один пример (0.0%), тогда как статьи упираются в лимит почти всегда — медиана `input_ids` равна 512, то есть 98.9% статей обрезаются. Ставить таргету такой же лимит, как входу, бессмысленно: почти все позиции ушли бы в паддинг, а память и время в self-/cross-attention растут квадратично по длине, поэтому лимиты и задают раздельно — 512 для энкодера (компромисс между покрытием статьи и стоимостью обучения) и 160 для декодера (с запасом относительно реальных саммари).
>
> Если поставить `MAX_SRC=64`, энкодер увидит только первые 2—3 предложения. Качество не упадёт до нуля — у новостей лид информативен, — но всё, что дальше (цифры, имена, развязка), станет для модели недоступным, а целевое саммари по-прежнему будет содержать эти факты. То есть на обучении мы прямо учим модель выдумывать то, чего нет во входе. Мои примеры это подтверждают даже на 512 токенах: в примере #163 эталон упоминает ~700 задержанных и список городов, которых в видимой части статьи нет, и модель вместо них подставила неверную дату митинга.


## 2. Бейзлайн (2 балла)

Прежде чем приступать к дообучению модели, необходимо зафиксировать бейзлайн на тестовой выборке. Это позволит оценить, насколько обучение действительно улучшило качество генерации.

Будут сравниваться две модели:

- **base**: `ai-forever/ruT5-base` **без дообучения**. Эта модель прошла только этап предобучения (pretraining) с использованием задачи восстановления замаскированных фрагментов текста (span corruption). Поэтому она не обучена выполнять суммаризацию и, как правило, будет генерировать саммари низкого качества. Именно этот результат и служит исходной точкой сравнения.
- **reference**: `IlyaGusev/rut5_base_sum_gazeta`, уже **дообученная** на задаче суммаризации новостей. Она используется в качестве эталонной модели, с результатами которой можно сравнить собственное решение. На небольшой обучающей выборке достичь такого же качества, скорее всего, не удастся, однако она позволяет понять, какого уровня можно ожидать от модели после полноценного обучения.

**Пояснение к метрикам.** ROUGE измеряет, насколько полно содержание сгенерированного саммари покрывает эталонный текст. Обычно анализируют ROUGE-1 (совпадение отдельных слов), ROUGE-2 (совпадение биграмм) и ROUGE-L (совпадение самой длинной общей подпоследовательности). BLEU, первоначально предложенная для машинного перевода, оценивает точность совпадения n-грамм предсказания с эталоном и сильнее штрафует за появление лишних слов и фраз.

> **Про ROUGE для русского языка.** Стандартный токенизатор библиотеки `rouge_score` рассчитан преимущественно на английский язык и некорректно обрабатывает кириллицу. В результате значения ROUGE для текстов на русском языке могут оказаться равными нулю. Поэтому в готовом коде используется `RougeScorer` с кастомным русским токенизатором. Не заменяйте его на `evaluate.load("rouge")` без соответствующей настройки токенизации, иначе рассчитанные значения метрики будут некорректными.

In [6]:
# Подсчет метрик с корректным русским токенизатором. НЕ МЕНЯЙТЕ токенизатор
class RuTokenizer:
    """Токенизатор для ROUGE"""
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=RuTokenizer())

def compute_metrics_text(preds, refs):
    """Принимает списки строк, возвращает dict с ROUGE-1/2/L и BLEU."""
    agg = {"rouge1": [], "rouge2": [], "rougeL": []}
    for p, r in zip(preds, refs):
        sc = _scorer.score(r, p)
        for k in agg:
            agg[k].append(sc[k].fmeasure * 100)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    return {"ROUGE-1": np.mean(agg["rouge1"]), "ROUGE-2": np.mean(agg["rouge2"]),
            "ROUGE-L": np.mean(agg["rougeL"]), "BLEU": bleu}

RESULTS = {}   # сюда складываем метрики каждой модели для итоговой таблицы

### ✍️ Задание 2.1. Генерация саммари и метрики бейзлайна

Реализуйте функцию `generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4)`: батчами токенизируйте `texts` (с `truncation`, `max_length=MAX_SRC`, `padding=True`, перенос на `device`), вызовите `model.generate(...)` с `num_beams`, `max_new_tokens=MAX_TGT` и `no_repeat_ngram_size=3`, декодируйте и верните список строк-саммари.

Затем посчитайте метрики для двух моделей на `test_texts` и сохраните в `RESULTS`:
- `RESULTS["reference"]` — для `REF_SUMMARIZER`;
- `RESULTS["base"]` — для `BASE_SEQ2SEQ` без дообучения.

In [7]:
# ✍️ ВАШ КОД (задание 2.1)
@torch.no_grad()
def generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4, **gen_kwargs):
    """Генерирует саммари для списка текстов батчами. Возвращает список строк."""
    model.eval()
    preds = []
    for start in tqdm(range(0, len(texts), batch_size), desc="generate"):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, truncation=True, max_length=MAX_SRC,
                        padding=True, return_tensors="pt").to(model.device)
        out = model.generate(
            **enc,
            num_beams=num_beams,
            max_new_tokens=MAX_TGT,
            no_repeat_ngram_size=3,   # запрет повторов: T5 любит зацикливаться
            **gen_kwargs,
        )
        preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    return preds


def evaluate_checkpoint(name, texts, refs, **kw):
    """Грузит чекпоинт по имени, генерирует саммари на texts и считает метрики.
    После подсчёта выгружает модель, чтобы не держать лишнее в памяти GPU."""
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    preds = generate_summaries(mdl, tok, texts, **kw)
    scores = compute_metrics_text(preds, refs)
    del mdl, tok
    torch.cuda.empty_cache()
    return scores, preds


In [8]:
# reference: уже дообученный суммаризатор (сильный ориентир)
RESULTS["reference"], ref_preds = evaluate_checkpoint(REF_SUMMARIZER, test_texts, test_refs)
print({k: round(v, 2) for k, v in RESULTS["reference"].items()})
print("\nПример генерации reference:\n", ref_preds[0])


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  828kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 1.31MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  977MB            

model.safetensors: downloading bytes:           |  0.00B            

generate:   0%|          | 0/38 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'ROUGE-1': np.float64(26.75), 'ROUGE-2': np.float64(10.57), 'ROUGE-L': np.float64(19.39), 'BLEU': 8.38}

Пример генерации reference:
 Бывший глава Белого дома Дональд Трамп заявил, что американские войска должны быть готовы вернуться в Афганистан при необходимости. Политик подчеркнул, что неважно, сколько американцев сейчас находятся в стране, а реальное число граждан США, оставшихся в республике, значительно превышает цифру, которую называет президент Джо Байден. Трамп отметил, что такого бы не произошло, если бы нынешние власти не придерживались разработанного его администрацией плана по выводу американских войск из Афганистана.


In [9]:
# base: ruT5-base без дообучения (прошёл только pretraining на span corruption)
RESULTS["base"], base_preds = evaluate_checkpoint(BASE_SEQ2SEQ, test_texts, test_refs)
print({k: round(v, 2) for k, v in RESULTS["base"].items()})
print("\nПримеры генерации base:")
for p in base_preds[:3]:
    print("-", repr(p[:200]))

display(pd.DataFrame(RESULTS).T.round(2))


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  892MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

generate:   0%|          | 0/38 [00:00<?, ?it/s]

{'ROUGE-1': np.float64(4.73), 'ROUGE-2': np.float64(0.27), 'ROUGE-L': np.float64(4.2), 'BLEU': 0.15}

Примеры генерации base:
- ', что в Афганистане заложницу.,'
- ', что такое, — это, — он, — замме. , — . ) . )))'
- ', в )))))))))))))))))))))))))'


,ROUGE-1,ROUGE-2,ROUGE-L,BLEU
reference,26.75,10.57,19.39,8.38
base,4.73,0.27,4.20,0.15


### 💬 Задание 2.2. Интерпретация результатов (впишите ответ ниже)

Используя полученные значения метрик, сравните качество базовой и референсной моделей. Насколько велик разрыв между ними? Объясните, почему базовая модель показывает такие результаты и зачем в машинном обучении вообще фиксируют бейзлайн, даже если заранее ожидают, что он будет значительно слабее.

> *Ваш ответ:* На одном и том же тесте (300 примеров из `test`) reference (`rut5_base_sum_gazeta`) даёт ROUGE-1 26.75, ROUGE-2 10.57, ROUGE-L 19.39, BLEU 8.38, а base (`ruT5-base` без дообучения) — ROUGE-1 4.73, ROUGE-2 0.27, ROUGE-L 4.20, BLEU 0.15. Разрыв не просто большой, а качественный: по ROUGE-1 — в 5.7 раза, по ROUGE-2 — в 39 раз, по BLEU — в 56 раз. Показательно, что сильнее всего проседают именно ROUGE-2 и BLEU: они считают совпадение биграмм и n-грамм, то есть требуют не «правильных слов», а связных фраз, и у base их практически нет.
>
> Причина в том, что base вообще не решала задачу суммаризации. Её единственная обучающая задача — восстановление замаскированных спанов (span corruption), поэтому на вход «статья → ?» она отвечает тем, что ей знакомо: обрывками фраз и мусором. Её реальные генерации выглядят так: `', что в Афганистане заложницу.,'`, `', что такое, — это, — он, — замме. , — . ) . )))'`, `', в )))))))))))))))))))))))))'`. Это не «плохая суммаризация», а отсутствие самой задачи в голове у модели: нет ни формата ответа, ни нужной длины, ни навыка сжатия. Reference же дообучена ровно на этом датасете и воспроизводит и стиль, и типичную длину саммари «Газеты» — её генерация на первом примере читается как готовый редакторский анонс.
>
> Бейзлайн фиксируют по трём причинам. Во-первых, это точка отсчёта: без неё нельзя утверждать, что прирост после дообучения — заслуга обучения, а не особенность среза данных. Во-вторых, это проверка пайплайна: если бы метрики считались неверно (например, стандартный английский токенизатор `rouge_score` зануляет кириллицу), это было бы видно сразу на бейзлайне, а не после часа обучения — ненулевые, но низкие 4.73 ROUGE-1 у base как раз показывают, что метрика работает, а модель — нет. В-третьих, пара base/reference задаёт коридор ожиданий: 4.73 — что даёт модель «из коробки», 26.75 — что даёт обучение на полном датасете, и свой результат осмысленно оценивать именно внутри этого коридора.


## 3. Дообучение seq2seq (2 балла)

На этом этапе вы дообучите модель `ruT5-base` на подвыборке обучающих данных. Цель задания — не получить наилучшее качество суммаризации, а проследить, как Fine-Tuning влияет на результаты модели. Поскольку обучение проводится всего на нескольких тысячах примеров и занимает одну эпоху, достигнуть качества референсной модели, обученной на полном датасете, не получится - и именно такой результат является ожидаемым.

> Если упираетесь в память или время: уменьшите `N_TRAIN` (в блоке 1), `MAX_SRC`, уменьшите `per_device_train_batch_size` или ограничьте обучение фиксированным числом шагов (`max_steps`) вместо полной эпохи.

> Про `fp16`. Для моделей семейства T5 обучение в `fp16` нередко становится нестабильным из-за переполнения значений активаций, что приводит к появлению `NaN` в loss. Ниже в конфигурации автоматически используется `bf16` (если его поддерживает оборудование), иначе обучение выполняется в `fp32`. Если во время обучения loss становится равным `NaN`, в первую очередь убедитесь, что `fp16` отключен.

In [10]:
# Модель, collator и функция метрик для Trainer. НЕ МЕНЯЙТЕ compute_metrics
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_SEQ2SEQ).to(device)
collator = DataCollatorForSeq2Seq(s2s_tok, model=model)

def compute_metrics(eval_pred):
    """Для Trainer: декодирует предсказания/таргет и считает ROUGE/BLEU."""
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, s2s_tok.pad_token_id)
    labels = np.where(labels != -100, labels, s2s_tok.pad_token_id)
    dec_preds = s2s_tok.batch_decode(preds, skip_special_tokens=True)
    dec_labels = s2s_tok.batch_decode(labels, skip_special_tokens=True)
    return {k: round(v, 2) for k, v in compute_metrics_text(dec_preds, dec_labels).items()}

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("bf16:", use_bf16)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

bf16: True


### ✍️ Задание 3.1. Дообучите модель

1. Допишите недостающие гиперпараметры в `Seq2SeqTrainingArguments` (см. комментарии в ячейке).
2. Создайте объект `Seq2SeqTrainer` и запустите обучение, вызвав `trainer.train()`.

Часть параметров уже задана (в том числе `predict_with_generate=True` и режим обучения `bf16`, если он поддерживается). Вам необходимо выбрать основные гиперпараметры обучения: размер батча, скорость обучения и продолжительность обучения.

In [12]:
# ✍️ ВАШ КОД (задание 3.1)
args = Seq2SeqTrainingArguments(
    output_dir="out_sum",
    predict_with_generate=True, # без этого eval не сгенерирует саммари и метрики будут пустыми
    bf16=use_bf16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    # --- гиперпараметры, которые выбираю сам ---
    per_device_train_batch_size=4,   # при MAX_SRC=512 надёжно влезает в 16 ГБ видеопамяти
    gradient_accumulation_steps=2,   # эффективный батч = 8 -> градиент менее шумный
    per_device_eval_batch_size=8,
    num_train_epochs=1,              # 3000 примеров / 8 ≈ 375 шагов: хватает, чтобы увидеть эффект
    learning_rate=3e-4,              # для T5 с AdamW типично 1e-4...5e-4; при NaN/скачках loss снизить до 1e-4
    lr_scheduler_type="linear",
    warmup_steps=0.05,               # короткий прогрев: в начале обучения градиенты самые «дикие»
    weight_decay=0.01,
    generation_max_length=MAX_TGT,   # eval генерирует с теми же лимитами, что и финальный инференс
    generation_num_beams=4,
    seed=RANDOM_STATE,
)


In [13]:
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],   # тест не трогаем: он только для финального сравнения
    data_collator=collator,                 # динамический паддинг + сдвиг labels в decoder_input_ids
    compute_metrics=compute_metrics,
)

try:
    trainer = Seq2SeqTrainer(**trainer_kwargs, processing_class=s2s_tok)
except TypeError:            # transformers < 4.46
    trainer = Seq2SeqTrainer(**trainer_kwargs, tokenizer=s2s_tok)

train_result = trainer.train()
print(train_result.metrics)


Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l,Bleu
1,5.473401,2.219260,26.370000,10.970000,20.270000,8.120000


{'train_runtime': 878.5986, 'train_samples_per_second': 3.415, 'train_steps_per_second': 0.427, 'total_flos': 1826873671680000.0, 'train_loss': 6.157834920247396, 'epoch': 1.0}


In [14]:
# метрики на валидации после обучения (тест по-прежнему не используется)
val_metrics = trainer.evaluate()
print({k: v for k, v in val_metrics.items() if "ROUGE" in k or "BLEU" in k or k == "eval_loss"})


Training Loss,Validation Loss,Epoch,Rouge-1,Rouge-2,Rouge-l,Bleu
5.473401,2.219260,1,26.370000,10.970000,20.270000,8.120000


{'eval_loss': 2.2192604541778564, 'eval_ROUGE-1': 26.37, 'eval_ROUGE-2': 10.97, 'eval_ROUGE-L': 20.27, 'eval_BLEU': 8.12}


## 4. Сравнение и анализ ошибок (3 балла)

Это центральный блок домашнего задания. Здесь вы не просто получите значения ROUGE и BLEU, а научитесь **интерпретировать эти метрики**, понимая, почему высокая оценка не всегда означает хорошее саммари, а низкая — не всегда плохое. Именно так обычно оценивают генеративные модели в реальных проектах.

### ✍️ Задание 4.1. Оценка дообученной модели и сравнение результатов

1. Сгенерируйте саммари **дообученной** моделью (`model` после `trainer.train()`) на `test_texts` и посчитайте метрики ROUGE и BLEU.
2. Сохраните полученные метрики в `RESULTS["fine-tuned"]`.
3. Выведите сводную таблицу по всем трем моделям: `pd.DataFrame(RESULTS).T.round(2)`.

In [15]:
# ✍️ ВАШ КОД (задание 4.1)
# та же функция генерации, тот же тест, те же метрики, что и для base/reference
ft_preds = generate_summaries(model, s2s_tok, test_texts, batch_size=8, num_beams=4)
RESULTS["fine-tuned"] = compute_metrics_text(ft_preds, test_refs)

table = pd.DataFrame(RESULTS).T.round(2)
table = table.loc[[m for m in ["base", "fine-tuned", "reference"] if m in table.index]]
display(table)

# строка с числами, которую удобно вставить в текстовые ответы
print(table.to_string())


generate:   0%|          | 0/38 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,ROUGE-1,ROUGE-2,ROUGE-L,BLEU
base,4.73,0.27,4.20,0.15
fine-tuned,25.23,9.69,18.33,6.16
reference,26.75,10.57,19.39,8.38


            ROUGE-1  ROUGE-2  ROUGE-L  BLEU
base           4.73     0.27     4.20  0.15
fine-tuned    25.23     9.69    18.33  6.16
reference     26.75    10.57    19.39  8.38


### ✍️ Задание 4.2. Анализ качества генерации

Выберите 2—3 примера из тестового набора и для каждого выведите:
- начало исходной статьи;
- эталонное саммари;
- саммари, сгенерированное дообученной моделью;
- значения ROUGE и BLEU для этого примера.

Сравните значения метрик со своей субъективной оценкой качества суммаризации. Есть ли примеры, где высокие метрики соответствуют неудачному саммари или, наоборот, низкие метрики получает вполне удачное саммари? Если да, кратко объясните, почему это произошло.

In [16]:
# ✍️ ВАШ КОД (задание 4.2)
def example_scores(i):
    """Метрики для одного примера (BLEU на одном предложении шумный — держим это в уме)."""
    return compute_metrics_text([ft_preds[i]], [test_refs[i]])


def show_example(i, n_chars=400):
    s = example_scores(i)
    print("=" * 110)
    print(f"[#{i}]  " + " | ".join(f"{k}={v:.2f}" for k, v in s.items()))
    print("\nСТАТЬЯ (начало):", " ".join(test_texts[i][:n_chars].split()), "...")
    print("\nЭТАЛОН        :", test_refs[i])
    print("\nFINE-TUNED    :", ft_preds[i])
    return s


# берём не случайные примеры, а показательные: лучший / средний / худший по ROUGE-2
r2 = np.array([example_scores(i)["ROUGE-2"] for i in range(len(ft_preds))])
order = np.argsort(r2)
picked = [int(order[-1]), int(order[len(order) // 2]), int(order[0])]
print("индексы (best / median / worst по ROUGE-2):", picked)

for i in picked:
    show_example(i)


индексы (best / median / worst по ROUGE-2): [110, 163, 295]
[#110]  ROUGE-1=55.91 | ROUGE-2=46.15 | ROUGE-L=53.76 | BLEU=39.33

СТАТЬЯ (начало): Минздрав России доработал и обновил временные методические рекомендации о вакцинации взрослого населения от COVID-19. В документе учли опыт применения препаратов для профилактики новой коронавирусной инфекции у беременных женщин и людей с онкологическими заболеваниями. Об этом в субботу, 24 июля, сообщили в пресс-службе ведомства. Рекомендации также доработаны в части порядка оформления медицински ...

ЭТАЛОН        : Методические рекомендации о вакцинации взрослого населения от COVID-19 доработали с учетом опыта применения препаратов для профилактики новой коронавирусной инфекции у беременных женщин и людей с онкологическими заболеваниями. В Минздраве разъяснили порядок получения временного медицинского отвода от вакцинации, а также указали противопоказания к прививке.

FINE-TUNED    : Минздрав России доработал и обновил временные методически

> *Комментарий к 4.2 (метрики vs субъективная оценка):*
>
> **Пример #110, ROUGE-2 = 46.15 (лучший по ROUGE-2).** Метрика завышает оценку. Первые два предложения моего саммари почти дословно повторяют лид статьи («Минздрав России доработал и обновил временные методические рекомендации…»), то есть модель не столько сжала текст, сколько скопировала его начало. Эталон при этом добавляет то, чего у меня нет (порядок получения временного медотвода и противопоказания), а моё третье предложение — про безопасность вакцин для беременных и детей — в эталон не входит вовсе. Саммари приемлемое, но высокий ROUGE-2 здесь — премия за совпадение формулировок с лидом, а не за качество сжатия.
>
> **Пример #163, ROUGE-2 = 7.41 (медианный).** Здесь метрика согласуется с моей оценкой, и саммари действительно слабое: модель уцепилась за первое предложение статьи (отключение мобильного интернета), верно вытащила факт про шумовые патроны у «Немиги», но придумала дату — «митинг 8 ноября» вместо воскресного митинга после выборов 9 августа — и не назвала главного, что есть в эталоне: около 700 задержанных в шести городах. Частично это следствие обрезки входа: 98.9% статей не помещаются в 512 токенов, и итоговые цифры задержаний, которые обычно стоят в конце новости, модель просто не видела.
>
> **Пример #295, ROUGE-2 = 0.00 (худший).** Здесь метрика занижает оценку, но не полностью ошибается. Саммари связное, грамматичное и по теме — «на глаз» оно явно лучше нуля; ROUGE-2 = 0 означает лишь, что с эталоном не совпала ни одна биграмма, потому что эталон пересказывает другую часть статьи (обращение к 18-му съезду и содержание доклада ЦК). Но при внимательном чтении видно настоящую ошибку, которую метрика как раз не диагностирует: модель перевернула роли — в статье рядовые коммунисты требуют, чтобы Зюганов ответил, а в саммари «Зюганов обратился к коллегам по партии с просьбой ответить». Отчасти это объяснимо: текст в датасете начинается со слов «Вместе с тем, по их мнению…», то есть местоимение «их» не имеет антецедента внутри входа.


### 💬 Задание 4.3. Интерпретация результатов

Проанализируйте полученные результаты.

1. Насколько дообученная модель улучшила качество по сравнению с базовой и насколько приблизилась к референсной? Подтвердите ответ значениями основных метрик.
2. Рассмотрите один из примеров из задания 4.2. Совпадает ли оценка качества по ROUGE с вашей субъективной оценкой саммари? Если нет - объясните, почему возникло расхождение. Если да — поясните, что именно хорошо отражает метрика в этом случае.

> *Ваш ответ:*
>
> 1. Одна эпоха на 3 тыс. примеров (375 шагов оптимизатора, ~14.5 минуты обучения) подняла метрики на том же тесте так: ROUGE-1 с 4.73 до 25.23, ROUGE-2 с 0.27 до 9.69, ROUGE-L с 4.20 до 18.33, BLEU с 0.15 до 6.16. От референсной модели (26.75 / 10.57 / 19.39 / 8.38) я отстал всего на 1.52 / 0.88 / 1.06 пункта ROUGE и на 2.22 пункта BLEU — то есть дообученная модель набрала ≈94% ROUGE-1 и ≈92% ROUGE-2 от reference, обученной на полном датасете (60 964 примера против моих 3 000). Честно говоря, я ожидал большего отставания, и этот разрыв стоит воспринимать осторожно: та же модель на валидации показала 26.37 ROUGE-1 против 25.23 на тесте, то есть разброс между срезами по 300 примеров — около одного пункта, что сопоставимо с самим разрывом. Заметнее ROUGE проседает BLEU (6.16 против 8.38): BLEU строже штрафует лишние слова и требует более длинных точных совпадений, а моя модель чаще «добирает» текст лишним предложением.
>
>    Качественная разница между base и fine-tuned даже больше, чем разница в числах, и полностью совпадает с моим впечатлением от текстов: base выдавала обрывки вроде `', в )))))))))))))))))))))))))'`, а после дообучения модель стабильно генерирует связное новостное саммари из 2—3 предложений в стиле «Газеты». То есть главное, чему модель научилась за одну эпоху, — это формат и жанр задачи, а не фактологическая точность: галлюцинации (неверная дата в #163, перевёрнутые роли в #295) никуда не делись.
>
> 2. Возьму пример #110 — тот, где ROUGE-2 максимальный (46.15 при ROUGE-1 = 55.91). По метрике это лучшее саммари теста, но с моей оценкой это совпадает лишь частично. Модель почти дословно воспроизвела первые два предложения статьи про обновлённые рекомендации Минздрава по вакцинации, а эталон те же факты переформулировал и добавил то, чего у меня нет, — порядок получения временного медотвода и противопоказания. Высокий ROUGE здесь награждает совпадение формулировок с лид-абзацем, то есть по сути экстрактивное копирование, а не сжатие и отбор главного. Для новостей это системная ловушка: лид написан информативно, ROUGE не отличает «выбрал главное» от «скопировал начало».
>
>    Обратное расхождение — пример #295 с ROUGE-2 = 0.00: саммари связное и по теме, но не совпало с эталоном ни одной биграммой, потому что эталон пересказывает другую часть статьи. Метрика сравнивает с одним-единственным эталоном и потому штрафует любой допустимый альтернативный пересказ; для русского это усугубляется морфологией, где «компания сообщила» и «компанией сообщено» — разные биграммы при одинаковом смысле. При этом настоящую ошибку в этом примере — инверсию субъекта («Зюганов обратился» вместо «к Зюганову обращаются») — ROUGE не диагностирует вовсе: ноль он поставил бы и за безупречный пересказ другими словами. А вот там, где метрика сработала честно, — пример #163: низкий ROUGE-2 = 7.41 действительно отражает, что модель потеряла ключевые факты эталона (≈700 задержанных, список городов) и добавила выдуманную дату.


## Блок 5. Выводы (1.5 балла)

### 💬 Задание 5.1. Выводы (впишите ответ ниже)

Подведите итоги выполненной работы (5–8 предложений).

В своем ответе отразите следующие вопросы:

- Что показало сравнение базовой, дообученной и референсной моделей? Дало ли дообучение заметный прирост качества и совпадает ли это с вашим впечатлением от сгенерированных саммари?
- Какую роль играет cross-attention в архитектуре seq2seq и почему без него декодеру было бы значительно сложнее формировать качественное саммари?
- Какие ограничения вы заметили у метрик ROUGE и BLEU? Какие способы оценки качества вы дополнительно использовали бы в реальном проекте?
- Какие идеи из этого задания напрямую переносятся на другие задачи условной генерации (машинный перевод, перефразирование)?

> *Ваш ответ:* Сравнение трёх моделей на одном фиксированном тесте дало ясную картину: base (ROUGE-1 4.73, ROUGE-2 0.27) задачу не решает вообще, одна эпоха дообучения на 3 тыс. примеров поднимает её до 25.23 / 9.69, а reference остаётся впереди с 26.75 / 10.57. Прирост огромный и совпадает с моим впечатлением от текстов, но важно, в чём именно он состоит: модель научилась формату — выдавать 2—3 связных предложения в новостном стиле вместо обрывков вроде `', в )))))))))))))))))))))))))'`, — а вот фактологическая точность осталась слабым местом (выдуманная дата в примере #163, перевёрнутые роли в #295). Отставание от reference оказалось меньше, чем я ожидал: 1.5 пункта ROUGE-1 при двадцатикратной разнице в объёме обучающих данных, причём такой же порядок расхождения я вижу между валидацией и тестом одной и той же модели (26.37 против 25.23), так что на 300 примерах этот разрыв близок к шуму.
>
> Cross-attention — это единственный канал, по которому декодер видит исходную статью: на каждом шаге он строит запрос из уже сгенерированного префикса и обращается к последовательности представлений энкодера, сам решая, какие позиции входа сейчас релевантны. Без него всю статью пришлось бы сжимать в один фиксированный вектор, и длинный текст неизбежно терял бы детали — именно те, которые нельзя «додумать»: имена, числа, даты. Мои примеры показывают это от обратного: там, где нужный факт физически отсутствовал во входе (98.9% статей обрезаются на 512 токенах, а в #295 текст вообще начинается с «Вместе с тем, по их мнению…» без антецедента), модель начинала правдоподобно выдумывать, потому что обращаться ей было не к чему.
>
> Главное ограничение ROUGE и BLEU я увидел на конкретных примерах: обе метрики сравнивают поверхностные n-граммы с одним эталоном, поэтому копирование лид-абзаца получает 46 ROUGE-2 (#110), а связный альтернативный пересказ — 0.00 (#295), и ни одна из них не заметила настоящей ошибки — инверсии субъекта. Для русского языка это усиливается морфологией, а BLEU на отдельных примерах ещё и нестабилен, так как задуман как корпусная метрика. В реальном проекте я бы оставил ROUGE/BLEU как дешёвый регрессионный сигнал между запусками, но решения принимал бы по метрикам на эмбеддингах (BERTScore), отдельной проверке фактологичности (NLI- или QA-based faithfulness: извлечь утверждения из саммари и проверить их по статье), LLM-as-a-judge с фиксированной рубрикой и небольшой ручной side-by-side разметке на нескольких десятках примеров как якорю для всего остального.
>
> Почти всё в этом задании переносится на другие задачи условной генерации без изменений: та же схема encoder–decoder, тот же loss по полю `labels`, те же раздельные лимиты длины для входа и выхода, тот же beam search на инференсе и та же схема сравнения «base / своя модель / сильный ориентир» на одном фиксированном тесте. Для машинного перевода и перефразирования меняются в основном данные и допустимая степень сжатия (там выход сопоставим по длине со входом, поэтому `MAX_SRC` и `MAX_TGT` сближаются), а BLEU и ROUGE точно так же измеряют пересечение n-грамм с одним эталоном — значит, и их слепые пятна наследуются целиком.


### 🤖 Использование генеративных нейросетей

*(обязательный пункт по условию ДЗ)*

**Цель использования.** Я использовал LLM (Claude) на двух этапах: (1) написание кода в ячейках `#ВАШ_КОД` и (2) поиск причины ошибки, возникшей при запуске. Содержательные решения — какие гиперпараметры выбрать, какие примеры разбирать, как трактовать расхождение метрик с моей оценкой — я принимал по результатам собственных прогонов ноутбука.

**Основные промпты:**

1. «Мне нужно решить ДЗ: дообучение seq2seq-модели (ruT5-base) для суммаризации новостей на датасете IlyaGusev/gazeta. Вот условие и ноутбук-шаблон — заполни ячейки `#ВАШ_КОД`: препроцессинг с раздельными `MAX_SRC`/`MAX_TGT`, функция генерации саммари батчами, аргументы `Seq2SeqTrainingArguments` и запуск `Seq2SeqTrainer`, подсчёт метрик и сводная таблица, разбор примеров.»
2. «При запуске ячейки 3.1 возникла ошибка `TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'`. Скажи, что исправить.» — ответ: в transformers v5 параметр `warmup_ratio` удалён, вместо него `warmup_steps`, принимающий float как долю шагов; заменил одну строку.

**Что я проверил и сделал сам.** Прогнал ноутбук целиком на GPU и убедился, что код работает и метрики осмысленны (base 4.73 ROUGE-1 против 26.75 у reference — значит, токенизация ROUGE для кириллицы считается корректно). Самостоятельно разобрал примеры #110, #163 и #295: читал статьи и эталоны и сверял их с генерацией — именно так я нашёл копирование лид-абзаца в #110 и инверсию субъекта в #295, которых в метриках не видно. Текстовые ответы (1.2, 2.2, 4.2, 4.3, 5.1) опираются на эти наблюдения и на числа из моих собственных прогонов.


## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

Это небольшой блок из двух коротких заданий на выбор — можно сделать одно или оба.

### ✍️ Задание Б1. Влияние стратегии декодирования

Сравните, как стратегия декодирования влияет на качество генерации.

1. Сгенерируйте саммари дообученной модели двумя способами:
    - beam search (`num_beams=4`, как в основной части);
    - жадное декодирование (`num_beams=1`, `do_sample=False`).
2. Посчитайте для обоих вариантов ROUGE и BLEU.
3. В 2–3 предложениях объясните, почему beam search обычно позволяет получить более высокие значения метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б1)
# beam search уже посчитан в 4.1 (ft_preds, num_beams=4)
greedy_preds = generate_summaries(model, s2s_tok, test_texts,
                                  batch_size=8, num_beams=1, do_sample=False)

decoding = {
    "beam search (num_beams=4)": compute_metrics_text(ft_preds, test_refs),
    "greedy (num_beams=1)": compute_metrics_text(greedy_preds, test_refs),
}
display(pd.DataFrame(decoding).T.round(2))

# средняя длина саммари в словах: beam search обычно чуть многословнее
for name, preds in [("beam", ft_preds), ("greedy", greedy_preds)]:
    print(name, "avg words:", round(np.mean([len(p.split()) for p in preds]), 1))

for i in picked[:1]:
    print("\nЭТАЛОН:", test_refs[i])
    print("BEAM  :", ft_preds[i])
    print("GREEDY:", greedy_preds[i])


> *Вывод (Б1):* Жадное декодирование на каждом шаге берёт самый вероятный токен и не может отыграть назад, поэтому одна ранняя неудачная развилка утягивает за собой всю последовательность. Beam search держит `num_beams` гипотез одновременно и сравнивает их по вероятности последовательности целиком, а не пословно, — так он находит варианты, которые начинаются менее вероятным токеном, но в сумме лучше. Для ROUGE/BLEU это ещё и удачно совпадает с устройством самих метрик: beam search тяготеет к более «усреднённым», частотным формулировкам, а именно они чаще попадают в n-граммы эталона.


### ✍️ Задание Б2. Что на самом деле измеряет ROUGE?

Проверьте, насколько ROUGE чувствителен к порядку слов.

1. Возьмите 3–5 эталонных саммари и случайным образом перемешайте слова в каждом из них (`random.shuffle`).
2. Посчитайте ROUGE-1 и ROUGE-2 между исходным и перемешанным текстом.
3. В 2–3 предложениях объясните:
    - почему ROUGE-1 обычно уменьшается незначительно;
    - почему ROUGE-2 падает гораздо сильнее;
    - что это говорит о том, какую информацию учитывает каждая из этих метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б2)
random.seed(RANDOM_STATE)

def shuffle_words(text):
    words = text.split()
    random.shuffle(words)
    return " ".join(words)

rows = []
for i in range(5):
    ref = test_refs[i]
    sc = _scorer.score(ref, shuffle_words(ref))   # эталон vs он же с перемешанными словами
    rows.append({"idx": i,
                 "ROUGE-1": sc["rouge1"].fmeasure * 100,
                 "ROUGE-2": sc["rouge2"].fmeasure * 100})

shuf = pd.DataFrame(rows).set_index("idx").round(2)
display(shuf)
print("среднее:", shuf.mean().round(2).to_dict())


> *Вывод (Б2):* ROUGE-1 при перемешивании слов почти не падает, потому что это пересечение мультимножеств униграмм: перестановка не меняет набор слов, а только их порядок, так что совпадение остаётся практически полным (расхождение от 100 даёт только нормировка по длине). ROUGE-2 обваливается почти до нуля, потому что биграмма — это пара соседних слов, и случайная перестановка разрушает все пары, кроме случайно уцелевших. Отсюда вывод: ROUGE-1 измеряет только лексическое покрытие — «те же ли слова использованы», — а хоть какую-то связность и порядок учитывают лишь ROUGE-2 и ROUGE-L. Полностью бессвязный набор правильных слов получит высокий ROUGE-1, поэтому судить о качестве генерации по одной этой метрике нельзя.
